# 02 · Fourth-Down Risk for an Underdog

**Audience:** Carolina Panthers head coach (hypothetical stakeholder)

**Question:** Given that the Carolina Panthers are projected underdogs in most of their remaining 2026 games, how much fourth-down risk should the head coach take?

**Goal:** Use historical NFL data to determine where going for it on fourth down pays off, whether the payoff is larger for underdogs, and how the Panthers' 2026 decisions compare.

## Context

*All facts below come from the data in Analysis A (nflverse pulled November 7, 2026; Panthers games through Week 8, league play-by-play through Thursday of Week 9).*

The Panthers are **3–4** (bye in Week 5), outscored 178–193. The betting market made them underdogs in 3 of their 7 games (1–2 in those: a win over Detroit, losses to Chicago and at Green Bay) and favorites in the other 4 (2–2).

Our Elo model rates Carolina at **1436** (ratings after the last completed game, November 5) and projects them as the **underdog in 7 of their 10 remaining games**. One is a large underdog spot (at MIN, 22% win probability, about 8.7 points), five are moderate (DEN, at TB, CIN, at PIT, SEA, 26–37%), and one is small (at NO, 47%). They are slight favorites only against BAL, NO at home, and ATL (52–59%). These are **model projections, not market lines**.

On fourth down, Carolina has gone for it 14 times in 47 fourth downs outside its own 20, converting 6 (42.9%, vs. 54.1% league-wide). In neutral situations it has gone for it on 7 of 29 fourth downs (24.1%), above the 2026 league rate of 16.6% (142 of 857), 7th of 32 teams. Both Panthers samples are under 30 plays, so they describe this season rather than establish a tendency.

## Methods

**Approach.** Every fourth down from the 2018–2025 regular seasons is labeled as *went for it* or *kicked*, and placed in a field-position zone and a yards-to-go group. Step B describes where attempts succeed and what happens after they succeed or fail. Step C asks whether going for it pays off more for underdogs, in two ways: (1) in neutral game situations, the average change in win probability (`vegas_wpa`) after going vs. kicking in the same zone and distance; (2) whether underdogs that were more aggressive on short fourth downs won more games than the betting market expected. Step D checks each Panthers 2026 fourth down against the evidence from C.

**Data.** nflverse via `nflreadpy` (the package the core model already uses): play-by-play and rosters for 2018–2026, schedules with pregame spreads and moneylines. Remaining 2026 Panthers games use **our Elo model's projections** (most recent ratings, `nflmodel.elo` with the tuned `DEFAULT_PARAMS`), not market lines.

**Locked definitions** (stored in `definitions.py`):

| Term | Definition |
|---|---|
| Underdog | Pregame win probability below 50%: market moneylines with the vig removed (historical and played 2026 games), or Elo-projected probability (remaining 2026 games). Games at exactly 50% or where the moneyline and spread disagree on the favorite are dropped from underdog/favorite comparisons. `spread_line` is the home team's expected margin (positive = home favored). |
| Spread groups | Small (1–3 points), moderate (3.5–7), large (7.5+), for underdogs and favorites. Elo lines are rounded to the nearest half point; under 1 point is "Pick'em". |
| Neutral situation | Quarters 1–3, score within 8 points, and `vegas_wp` between 10% and 90%. |
| Went for it | Run or pass on 4th down, excluding kneels, spikes, plays nullified by penalty, and aborted snaps from punt/field-goal formation. Fake punts/field goals count as attempts. |
| Kicked | Punt or field-goal attempt. |
| Converted | nflverse `fourth_down_converted` (first down or offensive touchdown). |
| Field zones (yards from opponent end zone) | Own 20–40 (61–80), Midfield (41–60), Opponent 40–21 (21–40), Red zone (6–20), Goal line (1–5). Fourth downs inside our own 20 are excluded. |
| Distance groups | 1, 2–3, 4–6, 7+ yards to go. |
| Play type (attempts) | QB sneak (designed QB run with 2 or fewer yards to go, not a scramble; QB from season roster), other run, pass. |
| Aggressiveness (team-game) | Neutral-situation go rate on 4th and 1–3 at or beyond midfield (`yardline_100` ≤ 50), team-games with at least 2 such plays. Low = 0%, Medium = between, High = 100%. |
| Wins over expected | Actual result (win 1, tie 0.5, loss 0) minus the no-vig market win probability. |
| Minimum sample | Any group under 30 plays (or team-games) is flagged "insufficient evidence" and no conclusion is drawn from it. For go-vs-kick comparisons, both sides need 30. |

**Scope of Step B.** Go rates use neutral situations only (so late-game desperation doesn't inflate them); conversion, EPA, drive results and cost of failure use all attempts outside the own 20.

## 0. Setup & data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import nflreadpy as nfl

from nflmodel.data import load_games, FRANCHISE_MAP
from nflmodel.elo import run_elo, DEFAULT_PARAMS, elo_diff, predicted_spread

from definitions import (HIST_SEASONS, CURRENT_SEASON, TEAM, MIN_SAMPLE,
                         no_vig_home_prob, spread_group, SPREAD_GROUP_ORDER,
                         is_neutral, decision, field_zone, distance_group, attempt_play_type,
                         AGGR_MAX_YDSTOGO, AGGR_MAX_YARDLINE, AGGR_MIN_PLAYS, aggression_group, AGGR_ORDER)

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

TABLES = Path("outputs/tables")
TABLES.mkdir(parents=True, exist_ok=True)


def save(table, name):
    """Write a summary table to outputs/tables/<name>.csv and return it for display."""
    table.to_csv(TABLES / f"{name}.csv", index=False)
    return table


def sample_flag(n):
    """Label groups below the minimum sample."""
    return np.where(n < MIN_SAMPLE, "insufficient evidence", "")

In [2]:
# 1. Play-by-play, regular season only (only the columns this analysis uses)
SEASONS = HIST_SEASONS + [CURRENT_SEASON]
PBP_COLS = ["game_id", "play_id", "season", "season_type", "week", "posteam", "defteam",
            "qtr", "down", "ydstogo", "yardline_100", "score_differential",
            "vegas_wp", "vegas_wpa", "epa", "play_type", "qb_kneel", "qb_spike", "aborted_play",
            "qb_scramble", "rusher_player_id", "fourth_down_converted", "fourth_down_failed",
            "fixed_drive", "fixed_drive_result", "posteam_score", "posteam_score_post", "desc"]
pbp = nfl.load_pbp(SEASONS).select(PBP_COLS).to_pandas()
pbp = pbp[pbp["season_type"] == "REG"].reset_index(drop=True)

# 2. Schedules (spreads, moneylines, scores); team codes mapped like the core games table (OAK -> LV)
schedules = nfl.load_schedules(SEASONS).to_pandas()
schedules = schedules[schedules["game_type"] == "REG"].copy()
schedules[["home_team", "away_team"]] = schedules[["home_team", "away_team"]].replace(FRANCHISE_MAP)

# 3. Rosters: which players were listed at QB each season (for QB sneaks)
rosters = nfl.load_rosters(SEASONS).to_pandas()
qbs = rosters[rosters["position"] == "QB"]
qb_ids = set(zip(qbs["season"], qbs["gsis_id"]))

latest_2026 = pbp[pbp["season"] == CURRENT_SEASON]
print(f"Plays loaded: {len(pbp):,} ({min(SEASONS)}-{max(SEASONS)} regular season)")
print(f"2026 play-by-play through Week {latest_2026['week'].max()} "
      f"({latest_2026['game_id'].nunique()} games)")

Plays loaded: 393,145 (2018-2026 regular season)
2026 play-by-play through Week 9 (122 games)


### Team-game table

One row per team per game, with the market win probability (vig removed), the team's line, underdog/favorite status, spread group, and result.

In [3]:
sch = schedules[["game_id", "season", "week", "home_team", "away_team", "home_score", "away_score",
                 "spread_line", "home_moneyline", "away_moneyline"]].copy()
sch["home_prob"] = no_vig_home_prob(sch["home_moneyline"], sch["away_moneyline"])

# Edge cases: exactly 50%, or the moneyline favorite is not the spread favorite
sch["edge_case"] = (np.isclose(sch["home_prob"], 0.5)
                    | ((sch["home_prob"] < 0.5) & (sch["spread_line"] > 0))
                    | ((sch["home_prob"] > 0.5) & (sch["spread_line"] < 0)))

home = sch.assign(team=sch["home_team"], opponent=sch["away_team"], is_home=True,
                  team_prob=sch["home_prob"], team_line=sch["spread_line"],
                  points_for=sch["home_score"], points_against=sch["away_score"])
away = sch.assign(team=sch["away_team"], opponent=sch["home_team"], is_home=False,
                  team_prob=1 - sch["home_prob"], team_line=-sch["spread_line"],
                  points_for=sch["away_score"], points_against=sch["home_score"])
team_games = pd.concat([home, away], ignore_index=True)

# Result: 1 win, 0.5 tie, 0 loss (NaN if not played yet)
played = team_games["points_for"].notna()
team_games["win"] = np.where(team_games["points_for"] > team_games["points_against"], 1.0,
                             np.where(team_games["points_for"] == team_games["points_against"], 0.5, 0.0))
team_games.loc[~played, "win"] = np.nan

# Status and spread group (blank for edge cases and games without a line)
has_line = team_games["team_prob"].notna() & ~team_games["edge_case"]
team_games["status"] = np.where(team_games["team_prob"] < 0.5, "Underdog", "Favorite")
team_games["spread_group"] = spread_group(team_games["team_line"])
team_games.loc[~has_line, ["status", "spread_group"]] = None

team_games = team_games[["game_id", "season", "week", "team", "opponent", "is_home", "points_for",
                         "points_against", "win", "team_prob", "team_line", "status", "spread_group", "edge_case"]]

hist_games = team_games[team_games["season"].isin(HIST_SEASONS)]
print(f"Historical games: {hist_games['game_id'].nunique():,}  |  edge-case games dropped from "
      f"underdog/favorite splits: {hist_games.loc[hist_games['edge_case'], 'game_id'].nunique()}")

Historical games: 2,127  |  edge-case games dropped from underdog/favorite splits: 15


### Fourth-down decisions

Label every fourth down, attach its zone, distance, neutral flag, and the team's pregame status.

In [4]:
fourth = pbp[pbp["down"] == 4].copy()
fourth["decision"], fourth["is_fake"], fourth["aborted_kick"] = decision(fourth)
fourth["zone"] = field_zone(fourth["yardline_100"])
fourth["distance"] = distance_group(fourth["ydstogo"])
fourth["neutral"] = is_neutral(fourth)

# What gets excluded, and why
hist4 = fourth[fourth["season"].isin(HIST_SEASONS)]
print("2018-2025 fourth-down plays:", f"{len(hist4):,}")
print("  not a counted decision (penalty no-plays, kneels, timeouts, etc.):", f"{hist4['decision'].isna().sum():,}")
print("    of which aborted snaps from punt/FG formation:", int(hist4["aborted_kick"].sum()))
print("  counted decisions inside own 20 (excluded):", int((hist4["decision"].notna() & hist4["zone"].isna()).sum()))

# Keep counted decisions outside our own 20
fourth = fourth[fourth["decision"].notna() & fourth["zone"].notna()].copy()
fourth["went"] = fourth["decision"] == "go"
fourth["converted"] = fourth["fourth_down_converted"] == 1

# Attempt play type (QB sneak / other run / pass)
fourth["play_group"] = None
fourth.loc[fourth["went"], "play_group"] = attempt_play_type(fourth[fourth["went"]], qb_ids)

# Pregame status of the team with the ball
fourth = fourth.merge(team_games[["game_id", "team", "status", "spread_group", "team_prob"]],
                      left_on=["game_id", "posteam"], right_on=["game_id", "team"], how="left")
assert fourth["team"].notna().all(), "every play should match a team-game"

hist = fourth[fourth["season"].isin(HIST_SEASONS)]
cur = fourth[fourth["season"] == CURRENT_SEASON]
fakes = hist[hist["is_fake"]]
print(f"\nCounted decisions 2018-2025: {len(hist):,}  (went for it {hist['went'].sum():,}, kicked {(~hist['went']).sum():,})")
print(f"  fake punts/FGs counted as attempts: {len(fakes)} "
      f"({fakes['desc'].str.contains('punt formation', case=False).sum()} from punt formation)")
print(f"  neutral-situation decisions: {hist['neutral'].sum():,}")

2018-2025 fourth-down plays: 31,985
  not a counted decision (penalty no-plays, kneels, timeouts, etc.): 1,946
    of which aborted snaps from punt/FG formation: 19
  counted decisions inside own 20 (excluded): 2590

Counted decisions 2018-2025: 27,449  (went for it 5,622, kicked 21,827)
  fake punts/FGs counted as attempts: 178 (156 from punt formation)
  neutral-situation decisions: 13,512


## A. Panthers 2026 context

In [5]:
# A1. Record and results so far (market lines for played games)
car = team_games[(team_games["team"] == TEAM) & (team_games["season"] == CURRENT_SEASON)].sort_values("week")
car_played = car[car["win"].notna()].copy()

wins, losses, ties = (car_played["win"] == 1).sum(), (car_played["win"] == 0).sum(), (car_played["win"] == 0.5).sum()
record = f"{wins}-{losses}" + (f"-{ties}" if ties else "")
print(f"{TEAM} {CURRENT_SEASON} record: {record}  |  points for {car_played['points_for'].sum():.0f}, "
      f"against {car_played['points_against'].sum():.0f}")
print(f"Market underdog in {(car_played['status'] == 'Underdog').sum()} of {len(car_played)} games played")

car_played["home_away"] = np.where(car_played["is_home"], "Home", "Away")
car_played["result"] = car_played["win"].map({1.0: "W", 0.5: "T", 0.0: "L"})
a1 = car_played[["week", "opponent", "home_away", "points_for", "points_against", "result",
                 "team_prob", "team_line", "status"]].rename(columns={"team_prob": "market_win_prob",
                                                                      "team_line": "market_line"})
save(a1, "A_car_2026_results").round(3)

CAR 2026 record: 3-4  |  points for 178, against 193
Market underdog in 3 of 7 games played


,week,opponent,home_away,points_for,points_against,result,market_win_prob,market_line,status
2129,1,CHI,Home,37.0,59.0,L,0.400,-3.0,Underdog
4543,2,ATL,Away,34.0,3.0,W,0.583,2.5,Favorite
4560,3,CLE,Away,18.0,21.0,L,0.551,2.5,Favorite
2189,4,DET,Home,32.0,26.0,W,0.336,-4.5,Underdog
4612,6,PHI,Away,25.0,14.0,W,0.593,3.0,Favorite
2224,7,TB,Home,13.0,43.0,L,0.600,3.0,Favorite
4633,8,GB,Away,19.0,27.0,L,0.369,-3.5,Underdog


In [6]:
# A2. Remaining games: Elo projections from our model's most recent ratings (NOT market lines)
games_all = load_games(2005, CURRENT_SEASON)          # fresh pull, kept in memory (core data file untouched)
elo = run_elo(games_all, DEFAULT_PARAMS)

last_played = elo[elo["home_win"].notna()]["kickoff"].max()
left = elo[(elo["season"] == CURRENT_SEASON) & (elo["game_type"] == "REG") & elo["home_win"].isna()
           & ((elo["home_team"] == TEAM) | (elo["away_team"] == TEAM))].copy()

# Unplayed games carry the latest ratings (Elo only updates on results), so these are current projections
car_home = left["home_team"] == TEAM
hfa = np.where(left["is_neutral"], DEFAULT_PARAMS["neutral_hfa"], DEFAULT_PARAMS["hfa"])
d_home = elo_diff(left["elo_home_pre"], left["elo_away_pre"], hfa)
home_margin = predicted_spread(d_home, DEFAULT_PARAMS["spread_divisor"])

remaining = pd.DataFrame({
    "week": left["week"].values,
    "opponent": np.where(car_home, left["away_team"], left["home_team"]),
    "home_away": np.where(car_home, "Home", "Away"),
    "car_elo": np.where(car_home, left["elo_home_pre"], left["elo_away_pre"]),
    "opp_elo": np.where(car_home, left["elo_away_pre"], left["elo_home_pre"]),
    "elo_win_prob": np.where(car_home, left["home_prob"], 1 - left["home_prob"]),
    "elo_proj_margin": np.where(car_home, home_margin, -home_margin),   # + = CAR favored by
})
remaining["elo_status"] = np.where(remaining["elo_win_prob"] < 0.5, "Underdog", "Favorite")
remaining["elo_spread_group"] = spread_group(remaining["elo_proj_margin"])
remaining["source"] = "Elo model projection"

print(f"Ratings as of the last completed game: {last_played:%Y-%m-%d}  |  CAR Elo {remaining['car_elo'].iloc[0]:.0f}")
print(f"Remaining games: {len(remaining)}  |  Elo-projected underdog in {(remaining['elo_status'] == 'Underdog').sum()}")
save(remaining, "A_car_2026_remaining_elo_projections").round(3)

Ratings as of the last completed game: 2026-11-05  |  CAR Elo 1436
Remaining games: 10  |  Elo-projected underdog in 7


,week,opponent,home_away,car_elo,opp_elo,elo_win_prob,elo_proj_margin,elo_status,elo_spread_group,source
0,9,DEN,Home,1435.686,1627.721,0.297,-6.001,Underdog,Underdog moderate (3.5-7),Elo model projection
1,10,NO,Away,1435.686,1413.997,0.471,-0.812,Underdog,Underdog small (1-3),Elo model projection
2,11,BAL,Home,1435.686,1466.226,0.516,0.458,Favorite,Pick'em (<1),Elo model projection
3,12,TB,Away,1435.686,1498.896,0.353,-4.208,Underdog,Underdog moderate (3.5-7),Elo model projection
4,13,MIN,Away,1435.686,1612.363,0.221,-8.747,Underdog,Underdog large (7.5+),Elo model projection
5,14,NO,Home,1435.686,1413.997,0.591,2.548,Favorite,Favorite small (1-3),Elo model projection
6,15,CIN,Home,1435.686,1568.014,0.373,-3.613,Underdog,Underdog moderate (3.5-7),Elo model projection
7,16,PIT,Away,1435.686,1574.499,0.261,-7.233,Underdog,Underdog moderate (3.5-7),Elo model projection
8,17,SEA,Home,1435.686,1614.954,0.312,-5.491,Underdog,Underdog moderate (3.5-7),Elo model projection
9,18,ATL,Home,1435.686,1461.849,0.523,0.633,Favorite,Pick'em (<1),Elo model projection


In [7]:
# A3. Panthers' 2026 fourth downs vs the 2026 league
def fourth_down_profile(df):
    """Attempts, conversions, and neutral-situation go rate for a set of fourth downs."""
    n = df[df["neutral"]]
    return pd.Series({
        "fourth_downs": len(df),
        "attempts": int(df["went"].sum()),
        "conversions": int(df["converted"].sum()),
        "conversion_rate": df.loc[df["went"], "converted"].mean(),
        "neutral_fourth_downs": len(n),
        "neutral_attempts": int(n["went"].sum()),
        "neutral_go_rate": n["went"].mean(),
    })

a3 = pd.DataFrame({
    TEAM: fourth_down_profile(cur[cur["posteam"] == TEAM]),
    "League (all 32 teams pooled)": fourth_down_profile(cur),
}).T.reset_index(names="group")
count_cols = ["fourth_downs", "attempts", "conversions", "neutral_fourth_downs", "neutral_attempts"]
a3[count_cols] = a3[count_cols].astype(int)
a3["attempts_flag"] = sample_flag(a3["attempts"])            # applies to conversion_rate
a3["neutral_flag"] = sample_flag(a3["neutral_fourth_downs"]) # applies to neutral_go_rate

# Where CAR ranks among the 32 teams on neutral go rate
by_team = cur.groupby("posteam").apply(fourth_down_profile, include_groups=False)
by_team["neutral_go_rate_rank"] = by_team["neutral_go_rate"].rank(ascending=False, method="min")
save(by_team.reset_index().sort_values("neutral_go_rate", ascending=False), "A_team_fourth_down_profile_2026")

print(f"CAR neutral go-rate rank: {by_team.loc[TEAM, 'neutral_go_rate_rank']:.0f} of {len(by_team)}")
save(a3, "A_car_vs_league_fourth_down_2026").round(3)

CAR neutral go-rate rank: 7 of 32


,group,fourth_downs,attempts,conversions,conversion_rate,neutral_fourth_downs,neutral_attempts,neutral_go_rate,attempts_flag,neutral_flag
0,CAR,47,14,6,0.429,29,7,0.241,insufficient evidence,insufficient evidence
1,League (all 32 teams pooled),1571,338,183,0.541,857,142,0.166,,


## B. Where and how teams succeed on fourth down (2018–2025)

In [8]:
# B1. Go rate by distance x zone (neutral situations)
neutral_hist = hist[hist["neutral"]]
b1 = (neutral_hist.groupby(["distance", "zone"], observed=True)
      .agg(fourth_downs=("went", "size"), attempts=("went", "sum"), go_rate=("went", "mean"))
      .reset_index())
b1["sample_flag"] = sample_flag(b1["fourth_downs"])
save(b1, "B_go_rate_by_distance_zone")

b1.pivot(index="distance", columns="zone", values="go_rate").round(3)

zone,Goal line (1-5),Red zone (6-20),Opponent 40-21 (21-40),Midfield (41-60),Own 20-40 (61-80)
distance,,,,,
1,0.912,0.822,0.840,0.758,0.258
2-3,0.358,0.260,0.418,0.283,0.029
4-6,0.081,0.026,0.170,0.080,0.014
7+,NaN,0.003,0.027,0.019,0.005


In [9]:
# B2. Conversion rate by distance, zone, play type, and distance x zone (all attempts)
attempts = hist[hist["went"]]

def conversion_table(by):
    t = (attempts.groupby(by, observed=True)
         .agg(attempts=("converted", "size"), conversions=("converted", "sum"), conversion_rate=("converted", "mean"))
         .reset_index())
    t["sample_flag"] = sample_flag(t["attempts"])
    return t

b2_distance = save(conversion_table(["distance"]), "B_conversion_by_distance")
b2_zone = save(conversion_table(["zone"]), "B_conversion_by_zone")
b2_play = save(conversion_table(["play_group", "distance"]), "B_conversion_by_play_type")
b2_cross = save(conversion_table(["distance", "zone"]), "B_conversion_by_distance_zone")

print(f"Attempts 2018-2025: {len(attempts):,}  |  overall conversion rate {attempts['converted'].mean():.3f}")
display(b2_distance.round(3), b2_zone.round(3), b2_play.round(3))
b2_cross.pivot(index="distance", columns="zone", values="conversion_rate").round(3)

Attempts 2018-2025: 5,622  |  overall conversion rate 0.539


,distance,attempts,conversions,conversion_rate,sample_flag
0,1,2274,1530,0.673,
1,2-3,1352,753,0.557,
2,4-6,975,453,0.465,
3,7+,1021,296,0.290,


,zone,attempts,conversions,conversion_rate,sample_flag
0,Goal line (1-5),643,355,0.552,
1,Red zone (6-20),807,431,0.534,
2,Opponent 40-21 (21-40),1711,909,0.531,
3,Midfield (41-60),1672,919,0.550,
4,Own 20-40 (61-80),789,418,0.530,


,play_group,distance,attempts,conversions,conversion_rate,sample_flag
0,Other run,1,1080,721,0.668,
1,Other run,2-3,221,141,0.638,
2,Other run,4-6,98,63,0.643,
3,Other run,7+,72,32,0.444,
4,Pass,1,591,326,0.552,
5,Pass,2-3,1093,589,0.539,
6,Pass,4-6,877,390,0.445,
7,Pass,7+,949,264,0.278,
8,QB sneak,1,603,483,0.801,
9,QB sneak,2-3,38,23,0.605,


zone,Goal line (1-5),Red zone (6-20),Opponent 40-21 (21-40),Midfield (41-60),Own 20-40 (61-80)
distance,,,,,
1,0.594,0.658,0.684,0.703,0.704
2-3,0.510,0.598,0.555,0.562,0.563
4-6,0.421,0.423,0.458,0.500,0.473
7+,NaN,0.246,0.276,0.301,0.316


In [10]:
# B3. Average EPA per attempt by distance (kicks shown for reference)
kicks = hist[~hist["went"]]
b3 = (attempts.groupby("distance", observed=True)
      .agg(attempts=("epa", "size"), avg_epa_attempt=("epa", "mean"), conversion_rate=("converted", "mean"))
      .join(kicks.groupby("distance", observed=True).agg(kicks=("epa", "size"), avg_epa_kick=("epa", "mean")))
      .reset_index())
b3["sample_flag"] = sample_flag(b3["attempts"])
save(b3, "B_epa_by_distance").round(3)

,distance,attempts,avg_epa_attempt,conversion_rate,kicks,avg_epa_kick,sample_flag
0,1,2274,0.259,0.673,1058,-0.386,
1,2-3,1352,0.432,0.557,2949,-0.111,
2,4-6,975,0.110,0.465,5388,-0.108,
3,7+,1021,-0.646,0.290,12432,0.019,


In [11]:
# B4. Drive results after a converted attempt (one row per converted attempt)
RESULT_MAP = {"Touchdown": "TD", "Field goal": "FG", "Punt": "Punt",
              "Turnover": "Turnover", "Opp touchdown": "Turnover",
              "Turnover on downs": "Turnover on downs"}
converted = attempts[attempts["converted"]].copy()
print(f"Converted attempts: {len(converted):,}")
converted["drive_result"] = converted["fixed_drive_result"].map(RESULT_MAP).fillna("Other (missed FG, end of half, safety)")

def result_shares(df, by):
    t = df.groupby(by + ["drive_result"], observed=True).size().rename("drives").reset_index()
    t["share"] = t["drives"] / t.groupby(by, observed=True)["drives"].transform("sum")
    t["group_total"] = t.groupby(by, observed=True)["drives"].transform("sum")
    t["sample_flag"] = sample_flag(t["group_total"])
    return t

b4_all = result_shares(converted.assign(all="All converted attempts"), ["all"]).rename(columns={"all": "zone"})
b4_zone = result_shares(converted, ["zone"])
b4 = save(pd.concat([b4_all, b4_zone], ignore_index=True), "B_drive_result_after_conversion")
b4.pivot(index="zone", columns="drive_result", values="share").round(3)

Converted attempts: 3,032


drive_result,FG,"Other (missed FG, end of half, safety)",Punt,TD,Turnover,Turnover on downs
zone,,,,,,
All converted attempts,0.188,0.077,0.053,0.522,0.081,0.078
Goal line (1-5),0.017,0.003,NaN,0.966,0.006,0.008
Midfield (41-60),0.233,0.103,0.078,0.371,0.110,0.104
Opponent 40-21 (21-40),0.255,0.099,0.024,0.501,0.054,0.067
Own 20-40 (61-80),0.153,0.077,0.160,0.280,0.177,0.153
Red zone (6-20),0.128,0.037,NaN,0.756,0.046,0.032


In [12]:
# B5. Cost of failure: the opponent's next drive after a failed attempt
# Drive table from every play: who had the ball, where the drive started, and points scored
plays = pbp[pbp["season"].isin(HIST_SEASONS) & pbp["fixed_drive"].notna() & pbp["posteam"].notna()]
plays = plays.sort_values(["game_id", "play_id"])
drives = (plays.groupby(["game_id", "fixed_drive"])
          .agg(drive_team=("posteam", "first"), first_play=("play_type", "first"),
               start_yardline_100=("yardline_100", "first"), score_start=("posteam_score", "first"))
          .reset_index())
end_score = (plays.groupby(["game_id", "fixed_drive", "posteam"])["posteam_score_post"].max()
             .rename("score_end").reset_index())
drives = drives.merge(end_score, left_on=["game_id", "fixed_drive", "drive_team"],
                      right_on=["game_id", "fixed_drive", "posteam"], how="left")
drives["points"] = drives["score_end"] - drives["score_start"]

# Failed attempt -> the next drive in the game, kept only if the opponent has the ball
# and the drive starts from scrimmage (not a kickoff, e.g. after halftime)
failed = attempts[~attempts["converted"]][["game_id", "fixed_drive", "defteam", "zone"]].copy()
failed["next_drive"] = failed["fixed_drive"] + 1
nxt = failed.merge(drives, left_on=["game_id", "next_drive"], right_on=["game_id", "fixed_drive"],
                   how="left", suffixes=("", "_next"))
matched = nxt[(nxt["drive_team"] == nxt["defteam"]) & (nxt["first_play"] != "kickoff")].copy()
print(f"Failed attempts: {len(failed):,}  |  matched to an opponent drive from scrimmage: {len(matched):,}")

def cost_table(df, by):
    t = (df.groupby(by, observed=True)
         .agg(failed_attempts=("points", "size"),
              opp_start_yards_to_end_zone=("start_yardline_100", "mean"),
              opp_avg_points=("points", "mean"),
              opp_td_rate=("points", lambda p: (p >= 6).mean()),
              opp_score_rate=("points", lambda p: (p > 0).mean()))
         .reset_index())
    t["sample_flag"] = sample_flag(t["failed_attempts"])
    return t

b5 = pd.concat([cost_table(matched.assign(zone="All failed attempts"), ["zone"]), cost_table(matched, ["zone"])],
               ignore_index=True)
save(b5, "B_cost_of_failure_next_drive").round(3)

Failed attempts: 2,590  |  matched to an opponent drive from scrimmage: 2,499


,zone,failed_attempts,opp_start_yards_to_end_zone,opp_avg_points,opp_td_rate,opp_score_rate,sample_flag
0,All failed attempts,2499,63.422,1.697,0.180,0.329,
1,Goal line (1-5),281,95.658,1.125,0.117,0.221,
2,Red zone (6-20),371,86.119,1.022,0.113,0.194,
3,Opponent 40-21 (21-40),790,67.005,1.656,0.182,0.310,
4,Midfield (41-60),707,50.161,2.083,0.221,0.403,
5,Own 20-40 (61-80),350,31.813,2.183,0.211,0.451,


## C. Does going for it pay off more for underdogs? (empirical approach)

> **Read with care: selection bias.** Teams choose when to go for it, and they tend to go when they like their chances (a good matchup, a short yard, a confident play call). Comparing the plays where teams went with the plays where they kicked therefore compares *different situations*, not the same situation decided two ways. The differences below are **associations, not causal effects** of going for it. The same applies to the team-game aggressiveness comparison: aggressive teams may differ in other ways (coaching, quarterback, game script) that also affect winning.

`vegas_wpa` is the change in the offense's win probability on the play (nflverse model that includes the pregame spread). Go minus kick > 0 means attempts in that cell were followed by larger win-probability gains than kicks. Neutral situations only; edge-case games dropped.

In [13]:
# C1. Go vs kick: average vegas_wpa by distance x zone, overall and split by status / spread group
neutral_split = neutral_hist[neutral_hist["status"].notna()]

def go_vs_kick(df, by):
    """Average vegas_wpa for go and kick, the difference, its standard error, and counts."""
    g = df.groupby(by + ["decision"], observed=True)["vegas_wpa"].agg(["size", "mean", "var"]).unstack("decision")
    t = pd.DataFrame({
        "go_plays": g[("size", "go")], "go_avg_wpa": g[("mean", "go")],
        "kick_plays": g[("size", "kick")], "kick_avg_wpa": g[("mean", "kick")],
    })
    t[["go_plays", "kick_plays"]] = t[["go_plays", "kick_plays"]].fillna(0).astype(int)
    t["go_minus_kick_wpa"] = t["go_avg_wpa"] - t["kick_avg_wpa"]
    t["std_error"] = np.sqrt(g[("var", "go")] / t["go_plays"] + g[("var", "kick")] / t["kick_plays"])
    t["evidence"] = np.where((t["go_plays"] >= MIN_SAMPLE) & (t["kick_plays"] >= MIN_SAMPLE),
                             "sufficient", "insufficient evidence")
    return t.reset_index()

c1_all = save(go_vs_kick(neutral_hist, ["distance", "zone"]), "C_go_vs_kick_wpa_all_teams")
c1_status = save(go_vs_kick(neutral_split, ["status", "distance", "zone"]), "C_go_vs_kick_wpa_underdog_favorite")
c1_spread = go_vs_kick(neutral_split, ["spread_group", "distance", "zone"])
c1_spread["spread_group"] = pd.Categorical(c1_spread["spread_group"], SPREAD_GROUP_ORDER, ordered=True)
c1_spread = save(c1_spread.sort_values(["spread_group", "distance", "zone"]), "C_go_vs_kick_wpa_spread_group")

print(f"Neutral decisions with a market status: {len(neutral_split):,} "
      f"(underdog {(neutral_split['status'] == 'Underdog').sum():,}, favorite {(neutral_split['status'] == 'Favorite').sum():,})")
c1_status.round(4)

Neutral decisions with a market status: 13,403 (underdog 6,938, favorite 6,465)


,status,distance,zone,go_plays,go_avg_wpa,kick_plays,kick_avg_wpa,go_minus_kick_wpa,std_error,evidence
0,Favorite,1,Goal line (1-5),95,0.0005,11,-0.0147,0.0152,0.0130,insufficient evidence
1,Favorite,1,Red zone (6-20),72,0.0131,15,-0.0023,0.0154,0.0100,insufficient evidence
2,Favorite,1,Opponent 40-21 (21-40),142,0.0132,22,-0.0189,0.0321,0.0124,insufficient evidence
3,Favorite,1,Midfield (41-60),169,0.0197,43,-0.0087,0.0284,0.0062,sufficient
4,Favorite,1,Own 20-40 (61-80),63,0.0211,166,-0.0042,0.0252,0.0083,sufficient
5,Favorite,2-3,Goal line (1-5),44,-0.0086,71,-0.0309,0.0224,0.0135,sufficient
6,Favorite,2-3,Red zone (6-20),35,0.0189,112,-0.0130,0.0319,0.0145,sufficient
7,Favorite,2-3,Opponent 40-21 (21-40),106,0.0054,134,-0.0150,0.0205,0.0093,sufficient
8,Favorite,2-3,Midfield (41-60),70,0.0125,184,-0.0042,0.0167,0.0092,sufficient
9,Favorite,2-3,Own 20-40 (61-80),4,0.0482,293,-0.0015,0.0497,0.0323,insufficient evidence


In [14]:
# C1 side by side: is the go-minus-kick payoff larger for underdogs in the same cell?
dog = c1_status[c1_status["status"] == "Underdog"].set_index(["distance", "zone"])
fav = c1_status[c1_status["status"] == "Favorite"].set_index(["distance", "zone"])
c1_compare = pd.DataFrame({
    "dog_go_plays": dog["go_plays"], "dog_kick_plays": dog["kick_plays"], "dog_go_minus_kick": dog["go_minus_kick_wpa"],
    "fav_go_plays": fav["go_plays"], "fav_kick_plays": fav["kick_plays"], "fav_go_minus_kick": fav["go_minus_kick_wpa"],
})
c1_compare["dog_minus_fav"] = c1_compare["dog_go_minus_kick"] - c1_compare["fav_go_minus_kick"]
c1_compare["both_sufficient"] = (dog["evidence"] == "sufficient") & (fav["evidence"] == "sufficient")
c1_compare = save(c1_compare.reset_index(), "C_underdog_vs_favorite_go_payoff")

ok = c1_compare[c1_compare["both_sufficient"]]
print(f"Cells with enough plays for both underdogs and favorites: {len(ok)} of {len(c1_compare)}")
print(f"  going favored (go minus kick > 0) for underdogs in {(ok['dog_go_minus_kick'] > 0).sum()}, "
      f"for favorites in {(ok['fav_go_minus_kick'] > 0).sum()}")
print(f"  payoff larger for underdogs than favorites in {(ok['dog_minus_fav'] > 0).sum()} of {len(ok)}")
c1_compare.round(4)

Cells with enough plays for both underdogs and favorites: 8 of 19
  going favored (go minus kick > 0) for underdogs in 7, for favorites in 8
  payoff larger for underdogs than favorites in 2 of 8


,distance,zone,dog_go_plays,dog_kick_plays,dog_go_minus_kick,fav_go_plays,fav_kick_plays,fav_go_minus_kick,dog_minus_fav,both_sufficient
0,1,Goal line (1-5),100,8,0.0390,95,11,0.0152,0.0238,False
1,1,Red zone (6-20),79,18,0.0177,72,15,0.0154,0.0023,False
2,1,Opponent 40-21 (21-40),130,30,0.0219,142,22,0.0321,-0.0102,False
3,1,Midfield (41-60),167,65,0.0238,169,43,0.0284,-0.0045,True
4,1,Own 20-40 (61-80),58,182,0.0214,63,166,0.0252,-0.0039,True
5,2-3,Goal line (1-5),38,75,0.0018,44,71,0.0224,-0.0205,True
6,2-3,Red zone (6-20),36,89,0.0419,35,112,0.0319,0.0101,True
7,2-3,Opponent 40-21 (21-40),83,128,0.0135,106,134,0.0205,-0.0069,True
8,2-3,Midfield (41-60),84,207,0.0273,70,184,0.0167,0.0106,True
9,2-3,Own 20-40 (61-80),14,312,0.0155,4,293,0.0497,-0.0342,False


In [15]:
# Spread-group view, sufficient cells only (full table incl. thin cells is in the CSV)
c1_spread[c1_spread["evidence"] == "sufficient"].round(4)

,spread_group,distance,zone,go_plays,go_avg_wpa,kick_plays,kick_avg_wpa,go_minus_kick_wpa,std_error,evidence
83,Underdog moderate (3.5-7),2-3,Opponent 40-21 (21-40),34,0.0353,47,0.0080,0.0273,0.0139,sufficient
98,Underdog small (1-3),1,Midfield (41-60),72,0.0231,33,0.0015,0.0216,0.0098,sufficient
102,Underdog small (1-3),2-3,Opponent 40-21 (21-40),30,-0.0006,49,-0.0080,0.0073,0.0182,sufficient
103,Underdog small (1-3),2-3,Midfield (41-60),32,0.0221,83,0.0026,0.0194,0.0142,sufficient
45,Favorite small (1-3),2-3,Opponent 40-21 (21-40),46,0.0143,67,-0.0110,0.0254,0.0144,sufficient
23,Favorite moderate (3.5-7),1,Own 20-40 (61-80),31,0.0191,71,-0.0067,0.0258,0.0109,sufficient
26,Favorite moderate (3.5-7),2-3,Opponent 40-21 (21-40),45,-0.0001,43,-0.0126,0.0125,0.0144,sufficient
27,Favorite moderate (3.5-7),2-3,Midfield (41-60),32,0.0239,71,-0.0047,0.0287,0.0134,sufficient


In [16]:
# C2. Team-game aggressiveness on 4th and 1-3 at or beyond midfield vs wins over expected
qual = neutral_split[(neutral_split["ydstogo"] <= AGGR_MAX_YDSTOGO) & (neutral_split["yardline_100"] <= AGGR_MAX_YARDLINE)]
tg = (qual.groupby(["game_id", "posteam"])
      .agg(qualifying_plays=("went", "size"), go_rate=("went", "mean"))
      .reset_index())
print(f"Team-games with any qualifying play: {len(tg):,}  |  with at least {AGGR_MIN_PLAYS}: "
      f"{(tg['qualifying_plays'] >= AGGR_MIN_PLAYS).sum():,}")
tg = tg[tg["qualifying_plays"] >= AGGR_MIN_PLAYS]

tg = tg.merge(team_games[["game_id", "team", "status", "spread_group", "team_prob", "win"]],
              left_on=["game_id", "posteam"], right_on=["game_id", "team"], how="left")
tg["aggression"] = pd.Categorical(aggression_group(tg["go_rate"]), AGGR_ORDER, ordered=True)
tg["wins_over_expected"] = tg["win"] - tg["team_prob"]

def woe_table(df, by):
    t = (df.groupby(by, observed=True)
         .agg(team_games=("win", "size"), avg_go_rate=("go_rate", "mean"),
              actual_win_rate=("win", "mean"), expected_win_rate=("team_prob", "mean"),
              wins_over_expected=("wins_over_expected", "mean"),
              woe_std_error=("wins_over_expected", lambda x: x.std() / np.sqrt(len(x))))
         .reset_index())
    t["sample_flag"] = sample_flag(t["team_games"])
    return t

c2_status = woe_table(tg, ["status", "aggression"])
c2_status = save(c2_status.sort_values(["status", "aggression"], ascending=[False, True]), "C_aggression_woe_underdog_favorite")

c2_spread = woe_table(tg, ["spread_group", "aggression"])
c2_spread["spread_group"] = pd.Categorical(c2_spread["spread_group"], SPREAD_GROUP_ORDER, ordered=True)
c2_spread = save(c2_spread.sort_values(["spread_group", "aggression"]), "C_aggression_woe_spread_group")

display(c2_status.round(3))
c2_spread.round(3)

Team-games with any qualifying play: 1,626  |  with at least 2: 409


,status,aggression,team_games,avg_go_rate,actual_win_rate,expected_win_rate,wins_over_expected,woe_std_error,sample_flag
3,Underdog,Low (0%),29,0.000,0.431,0.356,0.075,0.090,insufficient evidence
4,Underdog,Medium (1-99%),107,0.510,0.397,0.348,0.049,0.047,
5,Underdog,High (100%),79,1.000,0.405,0.341,0.064,0.054,
0,Favorite,Low (0%),15,0.000,0.733,0.649,0.084,0.118,insufficient evidence
1,Favorite,Medium (1-99%),113,0.497,0.739,0.637,0.102,0.041,
2,Favorite,High (100%),66,1.000,0.652,0.648,0.004,0.060,


,spread_group,aggression,team_games,avg_go_rate,actual_win_rate,expected_win_rate,wins_over_expected,woe_std_error,sample_flag
9,Underdog large (7.5+),Low (0%),6,0.000,0.333,0.229,0.104,0.213,insufficient evidence
10,Underdog large (7.5+),Medium (1-99%),18,0.523,0.194,0.196,-0.002,0.092,insufficient evidence
11,Underdog large (7.5+),High (100%),13,1.000,0.308,0.211,0.097,0.133,insufficient evidence
12,Underdog moderate (3.5-7),Low (0%),9,0.000,0.278,0.321,-0.043,0.148,insufficient evidence
13,Underdog moderate (3.5-7),Medium (1-99%),48,0.514,0.438,0.326,0.112,0.072,
14,Underdog moderate (3.5-7),High (100%),40,1.000,0.350,0.323,0.027,0.075,
15,Underdog small (1-3),Low (0%),14,0.000,0.571,0.433,0.138,0.139,insufficient evidence
16,Underdog small (1-3),Medium (1-99%),41,0.500,0.439,0.442,-0.003,0.079,
17,Underdog small (1-3),High (100%),26,1.000,0.538,0.433,0.106,0.098,insufficient evidence
6,Favorite small (1-3),Low (0%),8,0.000,0.750,0.557,0.193,0.159,insufficient evidence


## D. Panthers' 2026 decisions vs. the historical evidence

Each Panthers 2026 fourth down in a neutral situation is matched to its C1 cell (distance × zone × the Panthers' market status in that game). The evidence **favors going** when go minus kick > 0 and both sides have at least 30 plays; **favors kicking** when it is ≤ 0 with enough plays; otherwise it is **insufficient evidence**. Non-neutral fourth downs are counted separately and not classified.

In [17]:
car4 = cur[cur["posteam"] == TEAM].copy()
car4["opponent"] = car4["defteam"]

# Non-neutral plays: counted, not classified
non_neutral = car4[~car4["neutral"]]
print(f"{TEAM} 2026 fourth downs (outside own 20): {len(car4)}  |  neutral {car4['neutral'].sum()}  |  "
      f"non-neutral {len(non_neutral)} (went for it {non_neutral['went'].sum()}, kicked {(~non_neutral['went']).sum()})")
print(f"Games with no market status (edge case): {car4.loc[car4['status'].isna(), 'game_id'].nunique()}")

# Neutral plays: look up the historical go-minus-kick difference for the matching cell
evidence = c1_status[["status", "distance", "zone", "go_minus_kick_wpa", "go_plays", "kick_plays", "evidence"]]
car_n = car4[car4["neutral"]].merge(evidence, on=["status", "distance", "zone"], how="left")
car_n["evidence_says"] = np.select(
    [car_n["evidence"] != "sufficient", car_n["go_minus_kick_wpa"] > 0],
    ["Insufficient evidence", "Go favored"], "Kick favored")

d_plays = save(car_n[["week", "opponent", "status", "qtr", "score_differential", "ydstogo", "yardline_100",
                      "distance", "zone", "decision", "play_type", "converted", "go_minus_kick_wpa",
                      "go_plays", "kick_plays", "evidence_says", "desc"]].sort_values(["week", "qtr"]),
               "D_car_2026_neutral_fourth_downs")

d_summary = (car_n.groupby("evidence_says")
             .agg(plays=("went", "size"), went_for_it=("went", "sum"), go_rate=("went", "mean"))
             .reset_index())
d_summary["kicked"] = d_summary["plays"] - d_summary["went_for_it"]
d_summary = save(d_summary[["evidence_says", "plays", "went_for_it", "kicked", "go_rate"]], "D_car_2026_decision_summary")
d_summary.round(3)

CAR 2026 fourth downs (outside own 20): 47  |  neutral 29  |  non-neutral 18 (went for it 7, kicked 11)
Games with no market status (edge case): 0


,evidence_says,plays,went_for_it,kicked,go_rate
0,Go favored,11,5,6,0.455
1,Insufficient evidence,18,2,16,0.111


In [18]:
# Kicked plays where the historical evidence most favored going
d_kicked = (car_n[(car_n["evidence_says"] == "Go favored") & ~car_n["went"]]
            .sort_values("go_minus_kick_wpa", ascending=False)
            [["week", "opponent", "status", "qtr", "score_differential", "ydstogo", "yardline_100", "zone",
              "play_type", "go_minus_kick_wpa", "go_plays", "kick_plays"]])
save(d_kicked, "D_car_kicked_when_go_favored").round(4)

,week,opponent,status,qtr,score_differential,ydstogo,yardline_100,zone,play_type,go_minus_kick_wpa,go_plays,kick_plays
20,6,PHI,Favorite,1.0,3.0,1.0,58.0,Midfield (41-60),punt,0.0284,169,43
23,6,PHI,Favorite,3.0,6.0,6.0,54.0,Midfield (41-60),punt,0.0256,31,375
25,7,TB,Favorite,1.0,-7.0,6.0,56.0,Midfield (41-60),punt,0.0256,31,375
12,3,CLE,Favorite,3.0,-7.0,2.0,22.0,Opponent 40-21 (21-40),field_goal,0.0205,106,134
9,3,CLE,Favorite,1.0,0.0,2.0,56.0,Midfield (41-60),punt,0.0167,70,184
18,6,PHI,Favorite,1.0,0.0,4.0,29.0,Opponent 40-21 (21-40),field_goal,0.0030,57,241


In [19]:
# Cells the D classification could not use (thin historical samples)
car_n[car_n["evidence_says"] == "Insufficient evidence"].groupby(["status", "distance", "zone"], observed=True).agg(
    car_plays=("went", "size"), hist_go_plays=("go_plays", "first"), hist_kick_plays=("kick_plays", "first"))

car_plays  hist_go_plays  hist_kick_plays
status   distance zone                                                             
Favorite 1        Opponent 40-21 (21-40)          1            142               22
         7+       Red zone (6-20)                 1              1              448
                  Opponent 40-21 (21-40)          2             19              628
                  Midfield (41-60)                2             11              824
                  Own 20-40 (61-80)               3              6             1150
Underdog 1        Red zone (6-20)                 1             79               18
         4-6      Own 20-40 (61-80)               2             10              550
         7+       Opponent 40-21 (21-40)          3             17              669
                  Midfield (41-60)                1             23              946
                  Own 20-40 (61-80)               2              6             1331

In [20]:
# Thin cells across the analysis (listed in the Results Summary)
print("C1 underdog/favorite cells with insufficient evidence:")
print(c1_status[c1_status["evidence"] != "sufficient"][["status", "distance", "zone", "go_plays", "kick_plays"]].to_string(index=False))
print("\nC2 groups under 30 team-games:")
print(pd.concat([c2_status, c2_spread])[lambda t: t["sample_flag"] != ""]
      [["status", "spread_group", "aggression", "team_games"]].to_string(index=False))

C1 underdog/favorite cells with insufficient evidence:
  status distance                   zone  go_plays  kick_plays
Favorite        1        Goal line (1-5)        95          11
Favorite        1        Red zone (6-20)        72          15
Favorite        1 Opponent 40-21 (21-40)       142          22
Favorite      2-3      Own 20-40 (61-80)         4         293
Favorite      4-6        Goal line (1-5)         7          75
Favorite      4-6        Red zone (6-20)         8         255
Favorite      4-6      Own 20-40 (61-80)         5         473
Favorite       7+        Red zone (6-20)         1         448
Favorite       7+ Opponent 40-21 (21-40)        19         628
Favorite       7+       Midfield (41-60)        11         824
Favorite       7+      Own 20-40 (61-80)         6        1150
Underdog        1        Goal line (1-5)       100           8
Underdog        1        Red zone (6-20)        79          18
Underdog      2-3      Own 20-40 (61-80)        14         312


## Results Summary

**Panthers' context.** Carolina is 3–4 through Week 8, outscored 178–193, and was a market underdog in 3 of 7 games. Our Elo model (rating 1436 after the November 5 games) projects the Panthers as underdogs in 7 of their 10 remaining games, five of them by a moderate 3.5–7 points and one (at Minnesota, 22% win probability) by more than a touchdown. These are our model's projections, not market lines. So far Carolina has gone for it 14 times in 47 fourth downs outside its own 20 and converted 6 (42.9%, against a 2026 league rate of 54.1% on 338 attempts). In neutral situations it went for it on 7 of 29 fourth downs (24.1%, versus 16.6% league-wide on 857), 7th-most aggressive of 32 teams. Both Panthers samples are below the 30-play minimum.

**Where fourth-down attempts succeed (2018–2025).** Distance matters far more than field position. Across 5,622 attempts, teams converted 67.3% of fourth-and-1 tries (2,274 attempts), 55.7% at 2–3 yards (1,352), 46.5% at 4–6 (975) and 29.0% at 7 or more (1,021). Conversion rates by field zone sat in a narrow 53–55% band. The QB sneak was the most reliable play on fourth-and-1, converting 80.1% of 603 tries, compared with 66.8% for other runs (1,080) and 55.2% for passes (591). Average EPA per attempt was positive at 1 yard (+0.26), 2–3 yards (+0.43) and 4–6 yards (+0.11), and clearly negative at 7+ yards (−0.65). Of 3,032 converted attempts, 52.2% ended in a touchdown and 18.8% in a field goal; the rest ended in punts (5.3%), turnovers (8.1%), a later turnover on downs (7.8%) or other results (7.7%). After converting from midfield, the drive ended in a touchdown 37.1% of the time, and from the own 20–40 it was 28.0%. Failure has a real but bounded cost. Across 2,499 failed attempts matched to the opponent's next drive, the opponent scored on 32.9% of those drives and averaged 1.70 points. The cost was highest after failures in the own 20–40 (the opponent started about 32 yards from the end zone, scored on 45.1% and averaged 2.18 points, n = 350). It was lowest after failures at the goal line (the opponent started about 96 yards away and averaged 1.13 points, n = 281).

**Does going for it pay off more for underdogs?** In neutral situations (13,403 decisions with a usable market line), going for it was followed by a larger gain in win probability than kicking in nearly every cell with enough plays. That held for underdogs in 7 of 8 comparable distance-and-zone cells and for favorites in all 8. The clearest underdog results are 4th-and-1 at midfield (+2.4 win-probability points for going, 167 go vs. 65 kick plays, standard error 0.6), 4th-and-2–3 at midfield (+2.7, 84 vs. 207, SE 0.9), 4th-and-2–3 in the red zone (+4.2, 36 vs. 89, SE 1.5) and 4th-and-1 in the own 20–40 (+2.1, 58 vs. 182, SE 0.8). The exceptions were 4th-and-4–6 in the opponent's 40–21 (−0.9, 46 vs. 265) and 4th-and-2–3 at the goal line (+0.2, essentially zero). The evidence does not show a larger payoff for underdogs. The underdog go-minus-kick difference exceeded the favorite difference in only 2 of the 8 cells where both had enough plays: 2–3 yards in the red zone (+4.2 vs. +3.2) and at midfield (+2.7 vs. +1.7). The team-game test points the same way. Of 1,626 team-games with any neutral 4th-and-1–3 at or beyond midfield, 409 had at least two such plays. Among underdogs, the medium-aggression group (107 team-games) won 4.9 points more often than the market expected (SE 4.7) and the high-aggression group (79, always went for it) won 6.4 points more (SE 5.4). That gap is far smaller than its noise, and the low-aggression group (29 team-games) is too small to judge. Most spread-group splits fall under 30 team-games. The evidence therefore supports going for it in short-yardage spots at or beyond the own 20–40 *for any team*, but it does not show that being an underdog makes going for it more valuable.

**How the Panthers' 2026 decisions compare.** Of Carolina's 29 neutral-situation fourth downs, 11 fell in cells where the historical evidence (matched to their market status in that game) favored going for it. They went for it on 5 of those 11 (45%) and kicked on 6. The kicks where going was most favored were a Week 6 punt at Philadelphia on 4th-and-1 from their own 42 in the first quarter (cell difference +2.8 points), a Week 6 punt on 4th-and-6 from their own 46 in the third quarter (+2.6), a Week 7 punt against Tampa Bay on 4th-and-6 from their own 44 while trailing by 7 (+2.6), a Week 3 field goal at Cleveland on 4th-and-2 from the Browns' 22 while trailing by 7 in the third quarter (+2.1), a Week 3 punt on 4th-and-2 from their own 44 (+1.7), and a Week 6 field goal on 4th-and-4 from the Eagles' 29 (+0.3, effectively neutral). All six came in games where Carolina was the market favorite. The other 18 neutral fourth downs, mostly 7+ yards to go, fell in cells with too little evidence to classify (Carolina went for it on 2 of them). None fell in a cell where the evidence favored kicking. Another 18 non-neutral fourth downs (7 attempts, 11 kicks) were counted but not classified.

**Key limitations.** These are associations, not causal effects. Teams choose to go for it when they like their chances, so the go-versus-kick and aggressiveness comparisons are biased toward making going for it look better. The win-probability changes come from nflverse's `vegas_wpa` model, not observed outcomes. Many cells were too thin to use (fewer than 30 go or kick plays). For underdogs, these were 4th-and-1 at the goal line (8 kicks) and in the red zone (18 kicks); 2–3 yards in the own 20–40 (14 go); 4–6 yards at the goal line (7 go), in the red zone (4) and in the own 20–40 (10); and 7+ yards in the red zone (2 go), the opponent's 40–21 (17), midfield (23) and the own 20–40 (6). For favorites, they were 4th-and-1 at the goal line (11 kicks), in the red zone (15) and in the opponent's 40–21 (22 kicks); 2–3 yards in the own 20–40 (4 go); 4–6 yards at the goal line (7), in the red zone (8) and in the own 20–40 (5); and 7+ yards in the red zone (1), the opponent's 40–21 (19), midfield (11) and the own 20–40 (6). Most spread-group splits and the low-aggression groups are also below 30. The Panthers' own samples are small (14 attempts, 29 neutral fourth downs), and the remaining-game probabilities are Elo projections with ratings frozen after November 5. This analysis does not estimate the causal effect of going for it, does not model play-call quality or personnel, and does not include game-specific opponent defense. Those would need a different method, such as a win-probability decision model.